In [1]:
!pip install xgboost -q

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
from sklearn.base import clone
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

clf = XGBClassifier(objective="multi:softprob", eval_metric="mlogloss", random_state=42)
param_grid = {
    "n_estimators": [100, 300],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1],
}

In [2]:
train_df = pd.read_csv("/content/GSE98320.csv")
val_df   = pd.read_csv("/content/GSE129166.csv")

X_train = train_df.drop(columns=["sample_id", "diagnosis"])
y_train_raw = train_df["diagnosis"]
X_val   = val_df.drop(columns=["sample_id", "diagnosis"])[X_train.columns]
y_val_raw = val_df["diagnosis"]

In [3]:
imputer = SimpleImputer(strategy="median")
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_val   = pd.DataFrame(imputer.transform(X_val), columns=X_val.columns, index=X_val.index)

le = LabelEncoder()
y_train = pd.Series(le.fit_transform(y_train_raw), index=y_train_raw.index)
y_val   = pd.Series(le.transform(y_val_raw), index=y_val_raw.index)
class_labels = list(le.classes_)

In [4]:
def print_metrics(y_true_enc, y_pred_enc, label):
    y_true = le.inverse_transform(y_true_enc)
    y_pred = le.inverse_transform(y_pred_enc)
    print(f"\n=== XGBOOST — {label} ===")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=class_labels, zero_division=0)
    for cls, pi, ri, fi in zip(class_labels, p, r, f):
        print(f"  {cls:6s} | precision={pi:.4f}  recall={ri:.4f}  f1={fi:.4f}")
    print(f"  MACRO  | precision={np.mean(p):.4f}  recall={np.mean(r):.4f}  f1={np.mean(f):.4f}")

outer_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
X_train_r, y_train_r = X_train.reset_index(drop=True), y_train.reset_index(drop=True)
oof_pred = np.empty(len(y_train_r), dtype=int)

In [5]:
for fold, (tr_idx, te_idx) in enumerate(outer_cv.split(X_train_r, y_train_r)):
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    search = GridSearchCV(clone(clf), param_grid, cv=inner_cv, scoring="accuracy", n_jobs=-1)
    search.fit(X_train_r.iloc[tr_idx], y_train_r.iloc[tr_idx])
    oof_pred[te_idx] = search.predict(X_train_r.iloc[te_idx])
    print(f"  fold {fold+1}/10 done, best params={search.best_params_}")

print_metrics(y_train_r.values, oof_pred, "Cross-Validation Performance (GSE98320)")


  fold 1/10 done, best params={'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}
  fold 2/10 done, best params={'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}
  fold 3/10 done, best params={'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}
  fold 4/10 done, best params={'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300}
  fold 5/10 done, best params={'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100}
  fold 6/10 done, best params={'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300}
  fold 7/10 done, best params={'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 300}
  fold 8/10 done, best params={'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 300}
  fold 9/10 done, best params={'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}
  fold 10/10 done, best params={'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}

=== XGBOOST — Cross-Validation Performance (GSE98320) ===
Accuracy: 0.9187
  ABMR   | precision=0

In [6]:
final_search = GridSearchCV(clone(clf), param_grid,
                             cv=StratifiedKFold(3, shuffle=True, random_state=42),
                             scoring="accuracy", n_jobs=-1)
final_search.fit(X_train, y_train)
val_pred = final_search.predict(X_val)
print_metrics(y_val.values, val_pred, "Independent Validation Performance (GSE129166)")


=== XGBOOST — Independent Validation Performance (GSE129166) ===
Accuracy: 0.9351
  ABMR   | precision=0.7778  recall=0.9333  f1=0.8485
  NR     | precision=0.9825  recall=0.9333  f1=0.9573
  TCMR   | precision=1.0000  recall=1.0000  f1=1.0000
  MACRO  | precision=0.9201  recall=0.9556  f1=0.9352


In [8]:
val_mask = y_val != le.transform(["TCMR"])[0]
tcmr_code = list(le.classes_).index("TCMR")
val_mask = y_val != tcmr_code

X_val_no_tcmr = X_val[val_mask]
y_val_no_tcmr = y_val[val_mask]
val_class_labels = [c for c in class_labels if c != "TCMR"]

print(f"Excluded {(~val_mask).sum()} TCMR sample(s), evaluating on {len(y_val_no_tcmr)} remaining samples")

val_pred_enc = final_search.predict(X_val_no_tcmr)

# Decode back to string labels for readable metrics
y_true_dec = le.inverse_transform(y_val_no_tcmr.values)
y_pred_dec = le.inverse_transform(val_pred_enc)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

print(f"\n=== XGBOOST — Independent Validation Performance (GSE129166, TCMR excluded) ===")
print(f"Accuracy: {accuracy_score(y_true_dec, y_pred_dec):.4f}")
p, r, f, _ = precision_recall_fscore_support(y_true_dec, y_pred_dec, labels=val_class_labels, zero_division=0)
for cls, pi, ri, fi in zip(val_class_labels, p, r, f):
    print(f"  {cls:6s} | precision={pi:.4f}  recall={ri:.4f}  f1={fi:.4f}")
print(f"  MACRO  | precision={np.mean(p):.4f}  recall={np.mean(r):.4f}  f1={np.mean(f):.4f}")

Excluded 2 TCMR sample(s), evaluating on 75 remaining samples

=== XGBOOST — Independent Validation Performance (GSE129166, TCMR excluded) ===
Accuracy: 0.9333
  ABMR   | precision=0.7778  recall=0.9333  f1=0.8485
  NR     | precision=0.9825  recall=0.9333  f1=0.9573
  MACRO  | precision=0.8801  recall=0.9333  f1=0.9029
